# Helico: folding a protein from a contact map, no MSA

[Helico](https://github.com/Open-Athena/helico) is an AlphaFold3-style folding model that
takes a **residue–residue contact map** in place of a multiple sequence alignment.

On FoldBench, given contacts at the accuracy a current contact predictor delivers
(60% precision, 60% recall), it scores **0.824 lDDT** against **0.837** for
Protenix-with-MSAs — a difference indistinguishable from zero. With no contacts at all it
collapses to 0.316. So the contacts are doing the work the alignment used to do.

This notebook folds three proteins under four conditions and shows what the contact map
buys you.

> **Read this before believing the numbers.** The contacts here are computed from the
> *known* structure with [pyconfind](https://github.com/Open-Athena/pyconfind). That is
> the oracle condition — it uses the answer. It measures *structure realisation given a
> contact map*, not end-to-end prediction. Feeding real predicted contacts is
> [issue #11](https://github.com/Open-Athena/helico/issues/11), and is not done yet.

**Runtime:** Colab → *Runtime* → *Change runtime type* → **GPU**. An L4 or A100 is
comfortable; a T4 works but is slow and runs in float32 (no native bfloat16).

## 1. Install

In [ ]:
!pip install -q "helico @ git+https://github.com/Open-Athena/helico.git" py3Dmol
!pip install -q "cuequivariance-torch>=0.8,<0.9" "cuequivariance-ops-torch-cu12>=0.8,<0.9"

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > GPU"
name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
print(f"GPU: {name}  (compute capability {cc[0]}.{cc[1]})")
if cc[0] < 8:
    print("Note: pre-Ampere GPU — running in float32, which is slower. An L4/A100 is better.")

## 2. Load the model

Weights come from [timodonnell/helico](https://huggingface.co/timodonnell/helico) —
`contacts-msafree-01`, the checkpoint every number quoted above describes. It is MSA-free:
`use_msa=False` disables the MSA module *and* zeroes the alignment-derived conservation
features, so no alignment is fetched or used anywhere.

In [ ]:
from helico.inference import load_model
from helico.data import parse_ccd

model = load_model()          # downloads ~1.5 GB on first call
ccd = parse_ccd()             # chemical component dictionary
print(f"loaded on {model._helico_device} in {model._helico_dtype}")

## 3. Pick some proteins

Three small, well-behaved structures with different folds:

| PDB | what it is | length |
| --- | --- | --- |
| 1UBQ | ubiquitin — β-grasp fold | 76 |
| 1CRN | crambin — small and disulfide-rich | 46 |
| 1VII | villin headpiece — three-helix bundle, a folding-kinetics workhorse | 36 |

In [ ]:
import urllib.request
from pathlib import Path

from helico.bench import structure_to_chains
from helico.data import parse_mmcif, tokenize_sequences

TARGETS = {"1UBQ": "ubiquitin", "1CRN": "crambin", "1VII": "villin headpiece"}
Path("pdb").mkdir(exist_ok=True)

structures = {}
for pdb_id in TARGETS:
    path = Path("pdb") / f"{pdb_id}.cif"
    if not path.exists():
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.cif", path)
    gt = parse_mmcif(str(path))
    chains = structure_to_chains(gt)
    structures[pdb_id] = {
        "gt": gt,
        "chains": chains,
        "seq": chains[0]["sequence"],
        "tokenized": tokenize_sequences(chains, ccd),
    }
    print(f"{pdb_id} ({TARGETS[pdb_id]}): {len(chains[0]['sequence'])} residues")

## 4. Build contact maps

`contacts_from_structure` runs pyconfind on the known structure — the oracle map.

`contacts_from_pairs` is the path you would actually use in production: hand it a ranked
list of residue pairs from a contact predictor. Unlisted pairs stay **unknown**, never
"not a contact" — a truncated top-*n* list cannot tell those apart, and the model was
trained on exactly that convention.

In [ ]:
from helico.data import CONTACT_PRESENT
from helico.inference import contacts_from_structure

for pdb_id, s in structures.items():
    s["oracle"] = contacts_from_structure(s["tokenized"], s["gt"])
    n = int((s["oracle"] == CONTACT_PRESENT).sum()) // 2
    L = len(s["seq"])
    print(f"{pdb_id}: {n} contacts over {L} residues ({n/L:.2f} per residue)")

Here is the ubiquitin contact map. The diagonal band is absent by construction —
neighbours within 6 residues are excluded, since the chain already implies them and
pyconfind filters them out.

In [ ]:
import matplotlib.pyplot as plt
from helico.data import CONTACT_PRESENT

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (pdb_id, s) in zip(axes, structures.items()):
    ax.imshow((s["oracle"] == CONTACT_PRESENT).numpy(), cmap="Blues", interpolation="nearest")
    ax.set_title(f"{pdb_id} — {TARGETS[pdb_id]}", fontsize=10)
    ax.set_xlabel("residue"); ax.set_ylabel("residue")
plt.tight_layout(); plt.show()

## 5. Fold under four conditions

- **no contacts** — the model with nothing to go on
- **L/5 contacts** — a sparse list, roughly what a top-*n* predictor cut would give
- **60/60** — the true map degraded to 60% precision and 60% recall, i.e. a realistic
  predictor operating point
- **oracle** — the full true map, the ceiling

The 60/60 map reuses `sample_conditioning`, the same corruption used in training.

In [ ]:
import torch
from helico.contacts import sample_conditioning
from helico.data import CONTACT_PRESENT
from helico.inference import contacts_from_pairs, fold

def top_n_map(s, n):
    """Keep n of the true contacts, as a truncated top-n predictor list would."""
    ij = torch.triu((s["oracle"] == CONTACT_PRESENT), diagonal=1).nonzero()
    g = torch.Generator().manual_seed(0)
    keep = ij[torch.randperm(len(ij), generator=g)[:n]]
    return contacts_from_pairs([(int(i), int(j)) for i, j in keep],
                               tokenized=s["tokenized"])

def degraded(s, precision=0.6, recall=0.6):
    g = torch.Generator().manual_seed(0)
    return sample_conditioning(s["oracle"], generator=g, mode="contact-list",
                               precision=precision, recall=recall)

results = {}
for pdb_id, s in structures.items():
    L = len(s["seq"])
    conditions = {
        "no contacts": None,
        f"L/5 contacts (n={L//5})": top_n_map(s, L // 5),
        "60% prec / 60% recall": degraded(s),
        "oracle contacts": s["oracle"],
    }
    results[pdb_id] = {}
    for label, cmap in conditions.items():
        r = fold({"A": s["seq"]}, contacts=cmap, model=model, n_samples=2, ccd=ccd, seed=0)
        results[pdb_id][label] = r
        print(f"{pdb_id:6s} {label:26s} mean pLDDT {r.mean_plddt:5.1f}")
    print()

## 6. Score against the known structure

pLDDT is the model's *opinion* of itself. lDDT against the deposited structure is the
truth. They can disagree, so compute both.

In [ ]:
import numpy as np
import pandas as pd
from helico.bench import compute_lddt

def ca_coords_from_tokens(coords, tokenized):
    """CA coordinates in the model's atom order."""
    out, idx = [], 0
    for tok in tokenized.tokens:
        for name in tok.atom_names:
            if name == "CA":
                out.append(np.asarray(coords[idx]))
            idx += 1
    return out

def ca_coords_from_structure(structure):
    """CA coordinates from the deposited structure."""
    return [np.asarray(a.coords)
            for chain in structure.chains
            for res in chain.residues
            for a in res.atoms if a.name == "CA"]

def backbone_lddt(result, gt):
    """lDDT over CA atoms. Superposition-free, so no alignment step is needed."""
    pred = ca_coords_from_tokens(result.coords.numpy(), result.tokenized)
    truth = ca_coords_from_structure(gt)
    n = min(len(pred), len(truth))
    return compute_lddt(np.stack(pred[:n]), np.stack(truth[:n]))

# Sanity check the atom bookkeeping: scoring the deposited structure against
# itself must give exactly 1.0. If the CA ordering were wrong this would not.
_t = ca_coords_from_structure(structures["1UBQ"]["gt"])
assert abs(compute_lddt(np.stack(_t), np.stack(_t)) - 1.0) < 1e-6

rows = []
for pdb_id, per_cond in results.items():
    for label, r in per_cond.items():
        rows.append({"protein": pdb_id, "condition": label,
                     "mean pLDDT": round(r.mean_plddt, 1),
                     "lDDT vs deposited": round(backbone_lddt(r, structures[pdb_id]["gt"]), 3)})
df = pd.DataFrame(rows)
display(df.pivot(index="condition", columns="protein", values="lDDT vs deposited"))
df

In [ ]:
import matplotlib.pyplot as plt

order = ["no contacts"] + [c for c in df.condition.unique() if c.startswith("L/5")] + \
        ["60% prec / 60% recall", "oracle contacts"]
fig, ax = plt.subplots(figsize=(8, 4.2))
for pdb_id in results:
    sub = df[df.protein == pdb_id].set_index("condition").loc[order]
    ax.plot(range(len(order)), sub["lDDT vs deposited"], "-o", label=f"{pdb_id} ({TARGETS[pdb_id]})")
ax.set_xticks(range(len(order)))
ax.set_xticklabels([o.replace(" (", "\n(") for o in order], fontsize=8)
ax.set_ylabel("lDDT vs deposited structure"); ax.set_ylim(0, 1)
ax.grid(alpha=0.3, ls=":"); ax.legend(fontsize=9)
ax.set_title("Accuracy vs how much contact information the model is given", fontsize=11)
plt.tight_layout(); plt.show()

## 7. Look at the structures

Deposited structure in **grey**, prediction in **colour**. Toggle the condition to watch
the fold appear as contacts are supplied.

In [ ]:
import py3Dmol

def show(pdb_id, condition, width=760, height=460):
    gt_pdb = open(f"pdb/{pdb_id}.cif").read()
    view = py3Dmol.view(width=width, height=height)
    view.addModel(gt_pdb, "cif")
    view.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey", "opacity": 0.65}})
    view.addModel(results[pdb_id][condition].pdb, "pdb")
    view.setStyle({"model": 1}, {"cartoon": {"colorscheme": "chain"}})
    view.zoomTo()
    print(f"{pdb_id} — {condition}  (grey = deposited, colour = prediction)")
    return view.show()

show("1UBQ", "oracle contacts")

In [ ]:
show("1UBQ", "no contacts")

In [ ]:
show("1UBQ", "60% prec / 60% recall")

## 8. Feeding your own predicted contacts

This is the deployment path. `contacts_from_pairs` takes whatever your predictor ranks
highest:

```python
pairs = [(3, 41), (7, 65), ...]          # 0-indexed residue positions
contacts = contacts_from_pairs(pairs, tokenized=tokenized)
result = fold({"A": seq}, contacts=contacts, model=model)
```

For complexes, name the chains — inter-chain contacts are supported and carry no
sequence-separation restriction:

```python
pairs = [("A", 12, "B", 30), ("A", 15, "B", 27)]
contacts = contacts_from_pairs(pairs, tokenized=tokenized)
```

Use `one_indexed=True` if your predictor counts residues from 1. Pairs closer than 6
residues in sequence are dropped with a warning — pyconfind filters those from the ground
truth, so the model has never seen one asserted and its response is undefined.

### What to expect

The reported result is that at 60% precision and 60% recall the model matches
Protenix-with-MSAs, and that degrading a perfect map to that point costs only 0.012 lDDT —
the contact map is redundant enough that losing 40% of contacts and adding 40% false ones
is nearly free.

The important caveat: those false contacts were drawn *uniformly* by our noise model. Real
predictor errors cluster near true contacts, where they are geometrically plausible and
plausibly much harder to reject. Whether that changes the picture is exactly what
[issue #11](https://github.com/Open-Athena/helico/issues/11) is for.